<a href="https://colab.research.google.com/github/adams-x0/cv_project/blob/main/OWL_Vit_and_SAM_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🧰 STEP 1: Install dependencies
# ============================================================
!pip install transformers torch torchvision opencv-python matplotlib
!pip install git+https://github.com/facebookresearch/segment-anything.git

# Download SAM checkpoint (Base version)
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth


In [ ]:
# ============================================================
# 🦉 STEP 2: OWL-ViT Object Detection
# ============================================================
from transformers import OwlViTProcessor, OwlViTForObjectDetection
from PIL import Image
from IPython.display import display
import requests
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Load OWL-ViT processor and model
processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

# Load an example image
url = "https://images.unsplash.com/photo-1593642634315-48f5414c3ad9"
image = Image.open(requests.get(url, stream=True).raw)
display(image)

# Text prompts
texts = [["laptop", "person", "dog"]]  # You can change these labels

# Run inference
inputs = processor(text=texts, images=image, return_tensors="pt")
outputs = model(**inputs)

# Post-process results
target_sizes = torch.tensor([image.size[::-1]])  # (height, width)
results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes)[0]

# Show detections
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    if score > 0.3:
        print(f"Detected {texts[0][label]} with score {score:.2f} at {box}")

# Draw bounding boxes
plt.figure(figsize=(10,10))
plt.imshow(image)
ax = plt.gca()
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    if score > 0.3:
        x0, y0, x1, y1 = box.detach().cpu().numpy()
        rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, y0 - 5, f"{texts[0][label]} {score:.2f}", color='red', fontsize=12)
plt.show()


In [ ]:
# ============================================================
# ✂️ STEP 3: SAM Segmentation Using OWL-ViT Boxes
# ============================================================
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamPredictor

# Load SAM model
sam_checkpoint = "sam_vit_b_01ec64.pth"
model_type = "vit_b"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
predictor = SamPredictor(sam)

# Convert PIL to NumPy for SAM
image_np = np.array(image)
predictor.set_image(image_np)

# Extract boxes from OWL-ViT
boxes = results["boxes"].detach().cpu().numpy()

# Segment each detected object
masks_list = []
for box in boxes:
    masks, scores, logits = predictor.predict(box=box[None, :])
    mask = masks[0].astype(np.uint8) * 255
    masks_list.append(mask)

    # Overlay mask
    overlay = image_np.copy()
    overlay[mask > 0] = [0, 255, 0]  # green
    plt.imshow(overlay)
    plt.axis('off')
    plt.show()


In [ ]:
# ============================================================
# 💾 STEP 4: Optional — Export cutouts with transparency
# ============================================================
from PIL import Image as PILImage

for i, mask in enumerate(masks_list):
    alpha = mask
    rgba = np.dstack((image_np, alpha))
    PILImage.fromarray(rgba).save(f"object_{i}.png")
    print(f"Saved object_{i}.png")
